# Mathematical Basics - Part 2

This notebook covers the material in the last three chapters of the CWJSN Mathematical Basics section. Since much of this material is also covered in the Math_Foundations repo, I will only add interesting new stuff I learn along the way.

In [1]:
# Set up Revise.jl for automatic code reloading
# This only needs to be run once at the beginning of your notebook session
using Revise

# Load the package 
using Calculus
using Latexify

## Inequalities and Logical Expressions Chapter

### Geometric Mean expressed in Julia

It is not clear why Verzani chose in CWJSN to use the expression `(a * b)^(1/2)` in the example about the geometric mean, rather than `sqrt(a * b)` as the code for the expression $g = \sqrt{ab}$. A related question is whether `sqrt(a * b) == (a * b)^(1/2)` is always `true`. I asked Claude to provide some insight on this.

The geometric mean of $n$ numbers is

$$
G = \left(x_1 x_2 \cdots x_n\right)^{1/n}, \qquad n = 2:\ G = (ab)^{1/2} = \sqrt{ab},
$$

so the power form shows the definition and generalizes by changing the exponent. In the same CWJSN code block, the next line uses `sqrt` for the quadratic mean, so the choice may be stylistic.

| Measured (Julia 1.13.2, $10^7$ values of `rand()`) | Result |
| --- | --- |
| `x^(1/2) != sqrt(x)` | about 1.05% of values |
| largest difference | 1 ulp (unit in the last place) |
| example $x$ | `0.7092888978478461` |
| `x^(1/2)` | `0.8421929101149249` |
| `sqrt(x)` (correctly rounded) | `0.842192910114925` |

As for whether the two different expressions are `==`, running a test returns `false` for some inputs. With `a = 0.7092888978478461` and `b = 1.0` (chosen so the product will be equal to `x` in the table above):

```julia
sqrt(a * b) == (a * b)^(1/2)   # false
sqrt(a * b)                    # 0.842192910114925
(a * b)^(1/2)                  # 0.8421929101149249
```

Over a million pairs from `rand(2)`, 9,650 gave `false`, about 1 in 100.

**Why the two results differ:**

- **The exponent isn't the cause.** `1/2` is exactly `0.5` in binary, so `(a*b)^(1/2)` really does ask for the power 0.5.
- **`sqrt` always returns the closest floating-point number to the true root.** The IEEE floating-point standard requires that of square root, and the processor has an instruction for it.
- **`^` with a `Float64` exponent uses a general power routine.** It is built to be accurate to within about 1 unit in the last place for any exponent, but it doesn't promise the closest number. About 1% of the time it lands one step away from what `sqrt` gives.

Both results are within 1 unit in the last place of the true root, so `sqrt(a * b) ≈ (a * b)^(1/2)` is always `true`. It's the same `==` versus `≈` lesson as in the Part 1 section "Overflow, rounding and `≈`": two different routes to the same real number can land on adjacent floats.

### The bitwise operators

Verzani mentions bitwise operators but he doesn't explain them at all. He just says a few words about their precedence. So I asked Claude to write something up about bitwise operators. An extensively edited (by me) version follows.

#### What "bitwise" means

An integer is stored as a row of binary bits. For the purpose of bitwise operation, each bit is assigned a logical value. The bit `1` is equivalent to `true` and the bit `0` to `false`. Bitwise operators apply a logical function to the bits. Binary bitwise operators combine two rows of bits, operating on the two matched bits, position by position.

The bitwise operators `&` and `|` apply the truth value function of `and` and `or` respectively. Hence `&` puts a 1 only where both bits are 1, and `|` puts a 1 where either bit is 1.

Using the function `bitstring` that shows the bits in the rows, we get the following:

```julia
bitstring(Int8(12))        # "00001100"
bitstring(Int8(10))        # "00001010"
12 & 10                    # 8   = 00001000
12 | 10                    # 14  = 00001110
```
On truth values, `&` and `|` apply the same truth value function as `&&` and `||`, but they differ from them in four ways discussed below.


#### 1. Both sides are always evaluated

`&&` and `||` stop evaluating expressions as soon as the answer is known, which is why they are called shortcut (short-circuit) operators. `&` and `|` always evaluate both sides first.

A *guard* is a condition placed first so that the expression after it runs only when that is safe. In `x >= 0 && sqrt(x) > 1`, the test `x >= 0` guards `sqrt(x)`: when `x` is negative, `&&` already knows the answer is `false` and never calls `sqrt`. So a guard only works with `&&` (and `||`), not with `&` (or `|`):

```julia
x = -4
x >= 0 && sqrt(x) > 1        # false: sqrt is never called
(x >= 0) & (sqrt(x) > 1)     # DomainError with -4.0: sqrt runs anyway
```

#### 2. Precedence: above the comparisons, not below

`Base.operator_precedence` gives the levels (higher binds tighter):

```julia
Base.operator_precedence.([:&, :*, :|, :+, :<, :&&, :||])   # [12, 12, 11, 11, 7, 6, 5]
```

So `&` sits with `*` and `|` with `+`, both above the comparisons (`<` and the rest, 7), while `&&` and `||` sit below them.

`&&` and `||` bind more loosely than the comparisons, so `x > 0 && x < 10` needs no parentheses. `&` and `|` bind more tightly than the comparisons, so Julia joins `0 & x` before it compares anything:

```julia
x = 20
x > 0 & x < 10          # true  -- parsed as x > (0 & x) < 10, a chained comparison
(x > 0) & (x < 10)      # false -- the intended "0 < x < 10"
```

This gives no error, just a wrong answer. The same thing happens with CWJSN's $\lvert x-5\rvert > 7$:

```julia
x = 18
x - 5 < -7 | x - 5 > 7        # false -- parsed as x - 5 < ((-7 | x) - 5) > 7
(x - 5 < -7) | (x - 5 > 7)    # true
```

**Rule:** with `&` and `|`, put every comparison in parentheses.


#### 3. What the operands may be

`Bool` is Julia's type for truth values. It has exactly two values, `true` and `false`, and a comparison such as `3 > 2` returns one (`typeof(3 > 2)` is `Bool`). `Bool` is a subtype of `Integer`, with `true == 1` and `false == 0`, so arithmetic works on it (`true + true` is `2`). It is stored in a whole byte, not a single bit: `bitstring(true)` is `"00000001"`.

`&&` and `||` require a `Bool` on the left (`1 && true` is a TypeError: non-boolean (Int64) used in boolean context). The right side is returned as it is, so `true && 5` returns `5`. `&` and `|` are integer operations, so `true & 5` returns `1` (an `Int`), and `5 & 3.0` is a MethodError: there is no bitwise "and" for floats.

#### 4. Where `&` and `|` are the right tool: vectors

`&&` asks for one `true` or `false`, so it refuses a vector of them:

```julia
v = [-3, 2, 5, 12]
(v .> 0) && (v .< 10)       # TypeError: non-boolean (BitVector) used in boolean context
(v .> 0) .& (v .< 10)       # Bool[0, 1, 1, 0]
v[(v .> 0) .& (v .< 10)]    # [2, 5] -- keep the entries between 0 and 10
(v .< 0) .| (v .> 10)       # Bool[1, 0, 0, 1]
```
**What `v[(v .> 0) .& (v .< 10)]` does**

The expression listed in the code above needs some explanation. First, it is read from the inside out, with `v = [-3, 2, 5, 12]`. So step by step we see:

```julia
v .> 0                     # Bool[0, 1, 1, 1]  is each entry > 0?
v .< 10                    # Bool[1, 1, 1, 0]  is each entry < 10?
(v .> 0) .& (v .< 10)      # Bool[0, 1, 1, 0]  "and", applied entry by entry
v[Bool[0, 1, 1, 0]]        # [2, 5]            keep the entries marked true
```

1. **The dot broadcasts.** `v .> 0` compares each entry of `v` with 0 and gives one truth value per entry. The result is a `BitVector`, a vector of truth values packed one per bit, which prints as `Bool[…]`.
2. **`.&` applies "and" to each pair.** Position by position it combines the two vectors: true only where the entry is both > 0 and < 10.
3. **Indexing with a vector of truth values selects.** When the index is a vector of `true`/`false` the same length as `v`, `v[mask]` returns the entries where the mask is `true`, in order. This is called *logical indexing*. The mask must have the same length as `v`; `v[[true, false]]` is a BoundsError.

It is the same as finding the positions first and then indexing with them: `findall(mask)` gives `[2, 3]`, and `v[[2, 3]]` gives `[2, 5]`.

Two other ways to write the same selection:

```julia
v[0 .< v .< 10]                  # [2, 5]  the chained comparison, broadcast
filter(x -> 0 < x < 10, v)       # [2, 5]  keep the x for which the test is true
```
**Is it nonetheless possible to use `&&` on vectors? Only the dotted version: `.&&`**

Since Julia 1.7 there are also `.&&` and `.||`. They have the low precedence of `&&`, so `v .> 0 .&& v .< 10` is correct without parentheses (`Bool[0, 1, 1, 0]`).

Plain `&&` still refuses a vector. By contrast, the dotted `.&&` applies `&&` to each matched pair of entries. This is indicated by the Julia 1.7 release notes which state: "The short-circuiting operators `&&` and `||` can now be dotted to participate in broadcast fusion as `.&&` and `.||`." We can see this in action:

```julia
v = [-3, 2, 5, 12]
(v .> 0) && (v .< 10)      # TypeError: non-boolean (BitVector) used in boolean context
v .> 0 .&& v .< 10         # Bool[0, 1, 1, 0]
```

The short-circuit survives, entry by entry, because the whole dotted expression is fused into one loop:

```julia
x = [-4, 4]
x .>= 0 .&& sqrt.(x) .> 1          # Bool[0, 1] -- sqrt(-4) is never called
(x .>= 0) .& (sqrt.(x) .> 1)       # DomainError -- sqrt.(x) runs on every entry first
```

The Julia manual ("Noteworthy differences from other languages", MATLAB section) recommends `.&&` and `.||` for logic over arrays, e.g. `A .== 1 .|| A .== 2`.


#### `missing`

Both kinds of logical operators use the same three-valued logic, and the result is `missing` only when the unknown value could change the answer. `&` and `|` accept `missing` on either side, but `&&` and `||` accept it only on the right:

```julia
true  & missing      # missing -- depends on the unknown
false & missing      # false   -- false whatever the unknown is
true  | missing      # true
false | missing      # missing
missing && true      # TypeError: non-boolean (Missing) used in boolean context
true && missing      # missing -- the right side is returned as it is
```


#### The other bitwise operators

The manual ("Mathematical Operations", "Bitwise Operators") lists nine operators for integers: `~`, `&`, `|`, `⊻`, `⊼`, `⊽`, `>>>`, `>>`, `<<`. Using the same two rows as before (as `Int8`, so that `bitstring` shows 8 bits):

```julia
a, b = Int8(12), Int8(10)   # 00001100, 00001010
~a           # -13  = 11110011   not: flip every bit
a ⊻ b        #   6  = 00000110   xor: 1 where the bits differ
a ⊼ b        #  -9  = 11110111   nand: not (a & b)
a ⊽ b        # -15  = 11110001   nor:  not (a | b)
```

`⊻` is typed `\xor` Tab, `⊼` is `\nand` Tab and `⊽` is `\nor` Tab. Each one also has a name you can call: `xor(a, b)`, `nand(a, b)`, `nor(a, b)`.

Flipping every bit gives a negative number because of how integers store their sign (two's complement): `~x == -x - 1` for every integer, so `~5` is `-6`.

#### On truth values

```julia
~true            # false -- the same as !true
~5               # -6
!5               # MethodError: ! is only for Bool
true ⊻ true      # false -- "exactly one of them is true"
true ⊻ false     # true  (on truth values, ⊻ applies the same truth-value function as !=)
true ⊼ true      # false
false ⊽ false    # true
true ⊻ missing   # missing
```

So `!` is the logical "not" and only takes a `Bool`, while `~` is the bitwise "not" and takes any integer. Like `&` and `|`, all of these evaluate both sides and broadcast with a dot: `[true, false] .⊻ [true, true]` gives `Bool[0, 1]`, and `.~[true, false]` gives `Bool[0, 1]`.

#### Shifts

A shift slides the whole row of bits left or right. Bits that fall off the end are lost.

```julia
1 << 3              # 8   -- shifting left by n multiplies by 2^n
5 >> 1              # 2   -- shifting right by n divides by 2^n, rounding down
-16 >> 2            # -4  -- >> copies the sign bit into the new places (arithmetic shift)
Int8(-16) >>> 2     # 60  -- >>> fills them with 0s (logical shift): 11110000 → 00111100
Int8(64) << 1       # -128 -- the 1 slides into the sign bit: overflow wraps, as with *
1 << 64             # 0   -- every bit has fallen off an Int64
```


#### Precedence of all of them

```julia
Base.operator_precedence.([:^, :<<, :>>, :>>>, :*, :&, :⊼, :+, :|, :⊻, :⊽, :<])
# [15, 14, 14, 14, 12, 12, 12, 11, 11, 11, 11, 7]
```

So the shifts bind more tightly than `*` but less tightly than `^`, the "and" family (`&`, `⊼`) sits with `*`, and the "or" family (`|`, `⊻`, `⊽`) sits with `+`. In C the shifts bind more loosely than `+`, so code copied from C can change meaning: in Julia, `1 << 2 + 1` is `(1 << 2) + 1 = 5`, while in C it is `1 << 3 = 8`. All of these sit above the comparisons. Hence the rule from the previous section applies to every one of them: **_put each comparison in parentheses_**.

---

### CWJSN logical_expressions: Questions

Values marked * were not in my REPL session; they were added afterwards by Claude to complete the comparison. As always, try to answer on your own before looking at the answers.

| Q | Question (CWJSN) | My Julia → result | Answer | Why |
|---|---|---|---|---|
| 1 | Is `e^pi` or `pi^e` greater? | `ℯ^pi` → `23.140692632779267`; `pi^ℯ` → `22.459157718361038`; `ℯ^pi > pi^ℯ` → `true` | `e^pi` is greater | the two values differ by about 0.68, far more than any rounding error, so the `Float64` comparison can be trusted |
| 2 | Is $\sin(1000)$ positive? | `sin(1000)` → `0.8268795405320025`; `sin(1000) > 0` → `true` | Yes | 1000 radians is a whole number of turns plus `mod(1000, 2pi)` → `0.9735361584457891`\* radians, which lies in the first quadrant, where sine is positive |
| 3 | Suppose $0 < a < b$. How are $-1/a$ and $-1/b$ related? | `b = 10`, `a = 5`: `0 < a < b` → `true`; `1/a > 1/b` → `true`; `-1/b > -1/a` → `true` | $-1/a < -1/b$ | for positive numbers, taking reciprocals reverses the order ($1/a > 1/b$), and negating reverses it again |
| 4 | Suppose $a < 0 < b$. Is $1/a > 1/b$? | `a = -5` (`b = 10`): `a < 0 < b` → `true`; `1/b > 1/a` → `true` | It is never true | $1/a$ is negative and $1/b$ is positive, whatever the values of $a$ and $b$ |
| 5 | `airyai` is positive for $x > 0$. Which is the first negative value: `airyai(-1) < 0`, …, `airyai(-5) < 0`? | `for i in -5:-1 println(i,airyai(i) < 0) end` → `-5false`, `-4true`, `-3true`, `-2false`, `-1false` | `airyai(-3) < 0` | counting down from $-1$: $-1$ and $-2$ are still positive, and $-3$ is the first negative value. For negative $x$ the function oscillates, so $-5$ is positive again |
| 6 | By trying three values of $x > 0$: which could be always true, `x^x <= (1/e)^(1/e)`, `x^x == (1/e)^(1/e)` or `x^x >= (1/e)^(1/e)`? | `for i in rand()*(1:3) println((i,i^i >= (1/ℯ)^(1/ℯ))) end` → `true` three times ($i \approx 0.31, 0.62, 0.93$); the same loop with `==` → `false` three times; with `<=` → `false` three times | `x^x >= (1/e)^(1/e)` | $x^x$ has its minimum at $x = 1/e$, where it equals $(1/e)^{1/e} \approx 0.6922$. Values of $x$ below 1 are the real test: there $x^x < 1$, yet still $\ge 0.6922$ |
| 7 | Which is true for $x, y > 0$ and $0 < p < 1$: `(x+y)^p < x^p + y^p`, `==` or `>`? | `for i in 1:3,j=4:6,k = 0.5 println((i,j,k, (i+j)^k < i^k + j^k)) end` → `true` for all 9 pairs; the same loop with `>` → `false` for all 9; with `==` → `false` for all 9 | `(x+y)^p < x^p + y^p` | with $p = 1/2$, square both sides of $\sqrt{x+y} < \sqrt{x} + \sqrt{y}$: $x + y < x + y + 2\sqrt{xy}$, true whenever $x, y > 0$ |
| 8 | Which is always true for $a, b > 0$: `a^a + b^b <= a^b + b^a`, `a^a + b^b >= a^b + b^a` or `a^b + b^a <= 1`? | `for i in rand()*(1:3),j=rand()*(4:6) println((i,j, i^i + j^j >= i^j + j^i)) end` → `true` for all 9 pairs; the same loop with `<=` → `false` for all 9; with `==` → `false` for all 9 | `a^a + b^b >= a^b + b^a` | this is Ilani's inequality; the two sides are equal when $a = b$. The third option fails already at $a = b = 1$: `1^1 + 1^1 <= 1` → `false`\* |
| 9 | Is 3 in the set $\lvert x - 2\rvert < 1/2$? | `x = 3`; `abs(x-2) < 1/2` → `false` | No | $\lvert 3 - 2\rvert = 1$, which is not less than $1/2$ |
| 10 | Which is equivalent to $\lvert x - a\rvert > b$? | `x, a, b = 5, 3, 1`: `abs(x-a) > b` → `true`; `x, a, b = 5, 7, 1`: `abs(x-a) > b` → `true`, `-b > x -a` → `true` | $x - a < -b$ or $x - a > b$ | $\lvert y\rvert > b$ means $y$ lies more than $b$ away from 0, on either side: at $(5, 3, 1)$, $x - a = 2 > 1$; at $(5, 7, 1)$, $x - a = -2 < -1$. The other two options describe $\lvert x - a\rvert < b$, the opposite |
| 11 | If $\lvert x - \pi\rvert < 1/10$, is $\lvert \sin(x) - \sin(\pi)\rvert < 1/10$? | three runs of `x = pi + 1/10 * (2rand()-1)` ($x \approx 3.112, 3.167, 3.074$), each followed by `abs(x - pi) < 1/10, abs(sin(x) - sin(pi)) < 1/10` → `(true, true)` | True | $\sin\pi = 0$ and $\lvert\sin x\rvert = \lvert\sin(x - \pi)\rvert \le \lvert x - \pi\rvert < 1/10$, because $\lvert\sin u\rvert \le \lvert u\rvert$ for every $u$ |
| 12 | Does 12 satisfy $\lvert x - 3\rvert + \lvert x - 9\rvert > 12$? | `x = 12`; `abs(x -3) + abs(x-9) > 12` → `false` | No | $9 + 3 = 12$: the two sides are equal, so "greater than" is false |
| 13 | Which shows DeMorgan's law holds when both values are `false`? | see the code below the table | the third option | see the code below the table |
| 14 | For `Inf` and `NaN`: for which are both comparisons always `false`, `Inf < 3.0` and `3.0 <= Inf`, or `NaN < 3.0` and `3.0 <= NaN`? | `NaN < 3.0` → `false`; `3.0 <= NaN` → `false`; `Inf < 3.0` → `false`; `3.0 <= Inf` → `true` | `NaN < 3.0` and `3.0 <= NaN` | every comparison with `NaN` is `false`; `Inf` is greater than 3.0, so `3.0 <= Inf` is `true` |
| 15 | Do the four IEEE 754 facts hold in Julia: $+0$ equals but is not greater than $-0$; `Inf` equals itself and is greater than everything except `NaN`; `-Inf` likewise, less; `NaN` is not equal to, less than or greater than anything, itself included? | `-0.0 == 0.0` → `true`; `0.0 > -0.0` → `false`; `Inf == Inf` → `true`; `Inf > floatmax(Float64)` → `true`; `-Inf == -Inf` → `true`; `-Inf < -floatmax(Float64)` → `true`; `Inf > NaN`, `Inf == NaN`, `Inf < NaN` → `false`; `Inf != NaN`, `-Inf != NaN` → `true`; `NaN != NaN` → `true`; `NaN > NaN`, `NaN < NaN` → `false`; `NaN != nextfloat(0.0)` → `true` | Yes | `floatmax(Float64)` is the largest finite `Float64` and `nextfloat(0.0)` the smallest positive one, so they test the facts at the edges |
| 16 | `true && missing` gives `missing`, and the same with "or" gives `true`. Why each value? | see the code below the table | the first option, both times | see the code below the table |

**Q13 (DeMorgan's law)**, the three options:

```julia
!(false && false) == (!false && !false)   # true
!(false && false) == (false || false)     # false
!(false && false) == (!false || !false)   # true
```

Two options return `true` when both values are `false`, so this run alone cannot decide. DeMorgan's law is `!(a && b) == (!a || !b)`, the third option. The first is not a law: it fails when the values differ.

```julia
!(true && false) == (!true && !false)   # false *
```

**Q16 (`missing`)**:

```julia
true && missing, true || missing   # (missing, true)
```

- `true || missing` → `true`: in `a || b`, `b` is evaluated only if `a` is `false`. Here `a` is `true`, so `true` is returned and `missing` is never looked at.
- `true && missing` → `missing`: in `a && b`, `b` is evaluated only if `a` is `true`. Here it is, so `b` is evaluated and returned, and `b` is `missing`.

The second option for `||` claims `false || missing` would be `false`. It is not:

```julia
false || missing   # missing *
```


## Vectors and Containers Chapter

### More in-depth coverage of Vectors

Many of the topics covered in this chapter are covered in greater depth in my Linear Algebra repository. Below is a table that links topics in this CWJSN chapter to coverage in the documentation of that repo. Note that most of the topics are covered in the _Vectors_ chapter of the _Basics_ section, but some are covered in two chapters of the _Geometry_ section, which is currently less well edited. Links to the _Geometry_ section are clearly marked.

| Topic in the CWJSN vectors chapter | Linear Algebra section |
|---|---|
| A vector as an arrow with a magnitude and a direction | [What is a Vector?](https://fourm.info/linear_algebra/dev/Basics/01%20Vectors/#What-is-a-Vector%3F) |
| Components and the notation $\langle x,~y \rangle$ | [Vector Notation](https://fourm.info/linear_algebra/dev/Basics/01%20Vectors/#Vector-Notation) |
| Magnitude $\lVert \vec{v} \rVert = \sqrt{x^2 + y^2}$ | [Length (Magnitude) of a Vector](https://fourm.info/linear_algebra/dev/Basics/01%20Vectors/#Length-%28Magnitude%29-of-a-Vector) |
| Unit vector $\hat{v}$ | [Normalization and Unit Vectors](https://fourm.info/linear_algebra/dev/Basics/01%20Vectors/#Normalization-and-Unit-Vectors) |
| Julia's `norm`, and a unit vector as `v / norm(v)` | [Normalizing a Vector](https://fourm.info/linear_algebra/dev/Geometry/03%20Vectors%20and%20Geometry/#Normalizing-a-Vector) (Geometry) |
| Scalar multiplication $c\vec{v}$ | [Scalar Multiplication](https://fourm.info/linear_algebra/dev/Basics/01%20Vectors/#Scalar-Multiplication) |
| Vector addition $\vec{v} + \vec{w}$ | [Vector Addition](https://fourm.info/linear_algebra/dev/Basics/01%20Vectors/#Vector-Addition) |
| Vector subtraction $\vec{v} - \vec{w}$ | [Vector Subtraction](https://fourm.info/linear_algebra/dev/Basics/01%20Vectors/#Vector-Subtraction) |
| Standard unit vectors $e_x$, $e_y$ | [Standard Basis](https://fourm.info/linear_algebra/dev/Geometry/01%20Points%20and%20Coordinates/#Standard-Basis) (Geometry) |
| A vector written with any two non-parallel vectors | [Linear Combination](https://fourm.info/linear_algebra/dev/Basics/01%20Vectors/#Linear-Combination) |
| Polar form: the angle $\theta$ | [Polar Form of a Vector](https://fourm.info/linear_algebra/dev/Geometry/03%20Vectors%20and%20Geometry/#Polar-Form-of-a-Vector) (Geometry) |
| Finding the angle $\theta$ of $\langle x,~y \rangle$ with two-argument `atan` | [Recovering the Angle](https://fourm.info/linear_algebra/dev/Geometry/03%20Vectors%20and%20Geometry/#Recovering-the-Angle) (Geometry) |
| Higher dimensional vectors | [Components and Dimensions](https://fourm.info/linear_algebra/dev/Basics/01%20Vectors/#Components-and-Dimensions) |

### Julia Containers

#### Values, names and symbols

Below are some details about values, names and symbols created by Claude and edited by me. These three words come up throughout this section, so we will occasionally need to refer to the cells below to clarify related points. "The manual" referred to throughout the text below is the Julia manual.

##### Values

A *value* is what exists while a program runs: what a computation produces, what a name refers to and what a collection holds. The manual uses the word without giving it a formal definition, but it says two things about all values. In the chapter "Types":

> all values in Julia are true objects having a type

> Only values, not variables, have types – variables are simply names bound to values

Numbers and strings are values, but so are many other things that at first glance we wouldn't think of as data:

| Expression | Its value | `typeof` of the value |
|---|---|---|
| `5` | the number 5 | `Int64` |
| `2 + 3` | the number 5 | `Int64` |
| `2.5` | the number 2.5 | `Float64` |
| `"a"` | a string | `String` |
| `[1, 2]` | a vector | `Vector{Int64}` |
| `:x` | a symbol | `Symbol` |
| `sin` | the sine function | `typeof(sin)` |
| `Int64` | the type itself | `DataType` |

The first two rows show that a value is not the same as how it is written. This is the distinction between numbers and numerals from the _Mathematical Basics - Part 1_ notebook: `5` is a numeric literal, `2 + 3` is an expression, and both produce the same number. A value does not need a name either: `2 + 3` and `[10, 20, 30][2]` both produce a value without naming it.

Because the type belongs to the value, the same variable can refer to values of different types:

```julia
x = 5
typeof(x)      # Int64
x = "hello"
typeof(x)      # String
```

##### Names

A *name* is a word written in the source code. As we saw in the _Part 1_ notebook (in the "Numeric Literals" section), names are bound to values in various ways. The manual's chapter "Variables" says:

> A variable, in Julia, is a name associated (or bound) to a value.

When you run a program that has a stand-alone name, or type the name in the REPL, you get the value bound to the name. If nothing is bound to it, there is nothing to give: typing a (random) name `neverbound` in the REPL throws an `UndefVarError`.

The manual's chapter "Variables" separates assignment "(giving a new "name" to a value)" from mutation "(changing a value)". The "Can be changed" column in the table of collections below is about mutation:

```julia
a = [1, 2]
b = a          # two names, one value:
a === b        # true
push!(b, 3)    # mutation: changes the value
a              # [1, 2, 3]
a = [9]        # assignment: a now refers to a new value
b              # still [1, 2, 3]
```

##### Symbols

A *symbol* is a value of type `Symbol` that represents a name. The docstring for `Symbol` defines it as follows:

> The type of object used to represent identifiers in parsed julia code (ASTs). Also often used as a name or label to identify an entity (e.g. as a dictionary key).

The manual's chapter "Metaprogramming" calls it:

> an interned string used as one building-block of expressions

(See "Symbols are Interned strings" in the _Part 1_ notebook for what interned means.) You write one with `:`, or build one with `Symbol(...)`, which joins its arguments together:

```julia
:x                    # :x
typeof(:x)            # Symbol
Symbol("x") === :x    # true
Symbol("x", 1, :y)    # :x1y
```

Symbols are how Julia represents names in parsed code. `Meta.parse("x + 1").args` gives `[:+, :x, 1]`: the names `+` and `x` became symbols, and the numeric literal became the value `1`. When that code runs, the same chapter says:

> when an expression is evaluated, a symbol is replaced with the value bound to that symbol in the appropriate scope.

We can do that step ourselves with `eval`. Here `y` is a name, bound to a value which is the symbol `:x`:

```julia
x = 5
y = :x
y          # :x
eval(y)    # 5
```

A symbol does not need a variable behind it. In `linestyle = :dash` it is only a label. `s = :neverbound` works, and only `eval(s)` throws an `UndefVarError`. Its text does not need to be a valid name either: `Symbol("my key")` is a symbol (as can be seen by checking its `typeof`), though `my key` cannot be written as a plain name.

> ℹ️ <sub>**An aside:** You rarely need a symbol like this in your own code, because you choose your own keys and can leave out the space. It mostly comes up when the names come from someone else's data, as when a CSV file is read into a `DataFrame`. A column header such as `my key` becomes a column name that cannot be written as a plain name:</sub>
>
> <sub>
>
> ```julia
> using CSV, DataFrames
> df = CSV.read(IOBuffer("my key,other\n1,2\n3,4\n"), DataFrame)
> propertynames(df)          # [Symbol("my key"), :other]
> df[!, Symbol("my key")]    # [1, 3]
> df[!, "my key"]            # [1, 3]: a string works too
> ```
>
> </sub>
>
> <sub>To avoid such names altogether, `CSV.read` with `normalizenames = true` turns the header into `my_key`.</sub>

A symbol also does not pick out one particular variable, because scope decides that. `eval` works in global scope, so it does not see a local variable with the same name:

```julia
x = 5
let x = 99
    x          # 99
    eval(:x)   # 5
end
```

The docstring for `Symbol` also calls symbols immutable, although `ismutable(:x)` returns `true`, as does `ismutable("x")`. The docstring for `ismutable` says this is "For technical reasons".

##### Values, names and symbols side by side

In `x = 5`, the `x` is a name, the `5` gives the value, and `:x` is a separate value that represents the name.

| Question | Value | Name | Symbol |
|---|---|---|---|
| What is it? | What exists while the program runs | A word in the source code | A value that represents a name |
| Does it have a type? | Yes | No, only the value bound to it | Yes, `Symbol` |
| What does writing it give? | `5` gives 5 | `x` gives the value bound to it | `:x` gives `:x` |
| Can a collection hold it? | Yes | No: `[x]` holds the value of `x` | Yes: `[:x]` holds the symbol |
| Does it need a binding? | No | Yes, or `UndefVarError` | No |



##### What this means for collections

A collection holds values, never names. `[x, :x]` looks up `x` once and stores what it finds, so binding `x` to something else later does not affect the vector:

```julia
x = 5
v = [x, :x]    # Any[5, :x]
x = 6
v              # still Any[5, :x]
```

In a named tuple, the keys look like names, but no variable is created. They are stored as symbols, and they can only be symbols:

```julia
nt = (price = 10, qty = 2)
keys(nt)       # (:price, :qty)
nt[:price]     # 10
nt.price       # 10
price          # UndefVarError
("a" = 1,)     # syntax: invalid named tuple field name ""a""
```

A `Dict` has no such restriction, so symbols are only one choice of key, and one `Dict` can mix key types:

```julia
d = Dict(:price => 10, "qty" => 2)
d[:price]      # 10
d["qty"]       # 2
```

Destructuring by name, covered below, goes from the symbol back to a name: `(; price) = nt` looks up the key `:price` and binds the variable `price` to its value.

#### Collections vs Containers in Julia

Below are some details about Collections and Containers created by Claude and edited by me.

##### What Collections and Containers are

A *container* is a computer language value whose job it is to hold other values, its *elements*, and to hand them back. Verzani uses the term throughout this chapter but defines it only in passing. Where a programming language uses both words (Python, for one), *collections* narrows the term to refer to containers with a set of operations, including count, iterate, and often add and remove.

The Julia manual just talks about collections, and that is the term I personally have seen in all the books and articles I've read about Julia. In what follows we will stick to that, and refer to "collections", although Verzani uses both terms interchangeably. Rather than define the word, the manual's chapter "Collections and Data Structures" groups them by what they let you do (the operations): every collection can be iterated (`for x in c`) and has a `length`; some can be indexed by position (`c[i]`); dictionaries look a value up by a key; sets only answer whether a value is a member; and some can grow and shrink.

##### Characteristics of Collections

Julia collections differ in four ways: how you get an element out, whether the collection can be changed after it is made, whether its elements share one type, and whether it keeps its elements in order. The table compares the collections in this chapter on those four points. `SVector` and `MVector` come from the `StaticArrays.jl` package; the rest are in Base Julia.

| Collection | Example | Get an element by | Can be changed | Elements share one type | Keeps order |
|---|---|---|---|---|---|
| `Vector` | `[1, 2, 3]` | position: `v[2]` | values, and it can grow or shrink | yes | yes |
| `Matrix` | `[1 2; 3 4]` | row and column: `m[2, 1]` | values only | yes | yes |
| `Tuple` | `(1, "a")` | position: `t[2]` | no | no, each position has its own type | yes |
| `NamedTuple` | `(a = 1, b = 2)` | name: `nt.a`, or position: `nt[2]` | no | no, each name has its own type | yes |
| `Dict` | `Dict("a" => 1)` | key: `d["a"]` | values, and it can grow or shrink | keys share one type, values share one type | no |
| `Set` | `Set([1, 2])` | no access; test membership: `2 in s` | it can grow or shrink | yes | no |
| range (see notes) | `1:3` | position: `r[2]` | no | yes | yes |
| `String` | `"abc"` | byte position: `s[1]` | no | yes, every element is a `Char` | yes |
| `Pair` | `"a" => 1` | `p.first`, `p.second`, or position: `p[1]` | no | no, first and second have their own types | yes |
| `SVector` | `SVector(1, 2, 3)` | position: `s[2]` | no | yes | yes |
| `MVector` | `MVector(1, 2, 3)` | position: `m[2]` | values only | yes | yes |

Notes, all run on Julia 1.13.3 (StaticArrays 1.9.24):

- "Share one type" means one *element type*, `eltype`. It may be an abstract type: `[1, "a"]` is a `Vector{Any}`, while `[1, 2.0]` promotes both entries and is a `Vector{Float64}`.
- Grow or shrink: `push!`, `pop!`, `deleteat!` on a vector; `d["b"] = 2` and `delete!(d, "a")` on a dictionary; `push!` and `delete!` on a set.
- "Can be changed" describes what these operations do, not what `ismutable` reports, which can mislead both ways. `ismutable(Set([1, 2]))` is `false`, yet `push!` adds to it (see cell below about Sets). `ismutable("abc")` is `true`, yet a string has no operation that changes it (no `setindex!`, no `push!`).
- "range" is not a type name: ranges come in several types, all subtypes of `AbstractRange`. `1:3` is a `UnitRange`, `1:2:9` a `StepRange`, and `0:0.5:2` a `StepRangeLen`. Lowercase `range` is a function that builds one.
- A string is indexed by *byte* position, not by character count. In `"abc"` the two agree, but `α` takes two bytes, so in `"αβγ"` the characters sit at indices 1, 3 and 5, and `"αβγ"[2]` throws a `StringIndexError`.
- A dictionary or a set keeps neither the order you inserted nor sorted order: `Dict("c" => 1, "a" => 2, "b" => 3)` iterates `"a"`, `"c"`, `"b"`.
- Every collection here can be iterated, but `for x in c` does not always hand you what you might expect. A dictionary gives you `Pair`s: `for x in Dict("a" => 1)` sets `x` to `"a" => 1`, so write `for (k, v) in d` to get the key and the value separately. A string gives you `Char`s, one per character however many bytes it takes: `"αβγ"` occupies 6 bytes but `for x in "αβγ"` runs 3 times.
- A static vector's length is part of its type, so even an `MVector` cannot grow (see cell below about static vectors).

#### `Set`s

Below are some details about `Set`s created by Claude and edited by me.

##### Making a `Set`

The constructor `Set` takes one argument, an existing collection (or anything else that can be iterated over), and builds a new set from its elements. You can't construct a set by passing the elements as separate arguments: `Set(1, 2)` is a `MethodError`. When constructing the set from the collection, duplicates collapse to one element and the order of the source collection is dropped, since a set keeps no order. The source collection is not changed; the set is a new object.

Taking a collection rather than separate elements is what lets `Set` turn any collection we already have—a vector in a variable, a range, a string—into a set. It also means a collection given to `Set` is always read as the *source* of the elements, never as an element itself: a set whose one element is a vector needs the vector wrapped in another collection.

The other way to make a set is to start with an empty one and add elements with `push!`. Give the element type in braces, `Set{Int}()`; a bare `Set()` has element type `Any`.

| Call | Result | What happens |
|---|---|---|
| `Set([1, 2])` | `Set([2, 1])` | the elements of a vector |
| `Set((1, 2))` | `Set([2, 1])` | the elements of a tuple |
| `Set(1:3)` | `Set([3, 2, 1])` | the elements of a range |
| `Set("abba")` | `Set(['b', 'a'])` | the characters of a string, duplicates collapsed |
| `Set(v)` with `v = [1, 2, 2, 1]` | `Set([2, 1])` | duplicates collapsed; `v` is still `[1, 2, 2, 1]` |
| `Set([[1, 2]])` | `Set([[1, 2]])` | one element, the vector `[1, 2]` |
| `Set(1, 2)` | `MethodError` | no method takes the elements as separate arguments |
| `Set(1)` | `Set([1])` | works only because a number iterates as itself, a one-element collection |
| `Set{Int}()`, then `push!` 1, 2, 1 | `Set([2, 1])` | start empty, add elements; the second 1 is already there |
| `Set()` | `Set{Any}()` | empty, element type `Any` |

`Dict` differs here: it takes its pairs either way, as separate arguments, `Dict("a" => 1, "b" => 2)`, or as one collection, `Dict(["a" => 1, "b" => 2])`.

All run on Julia 1.13.3.

##### How a `Set` is stored

The two rules above, duplicates collapse and order is dropped, come from how Base Julia stores a set. This is an implementation detail: using a set never requires understanding this, which is why I placed this section after the previous one. However, how it is stored explains its behavior, so if you are interested, the details follow.

Base (`base/set.jl`) defines `Set` as a `struct` with one field, a `Dict`:

```julia
struct Set{T} <: AbstractSet{T}
    dict::Dict{T,Nothing}
end
```

The elements of the set are the *keys* of that dictionary. Every value is `nothing`, so only the keys carry information:

```julia
s = Set([1, 2, 2, 1])
s.dict                 # Dict(2 => nothing, 1 => nothing)
```

This is where the two rules come from. Duplicates collapse because a dictionary cannot hold the same key twice: adding `2` again only sets `dict[2] = nothing` once more. Order is dropped because a `Dict` is a hash table: it places each key by its hash, not by when it was added.

Every set operation is a dictionary operation underneath, and `Set(itr)` itself starts from an empty dictionary and adds each element of `itr` as a key:

| Set operation | What Base does |
|---|---|
| `push!(s, x)` | `s.dict[x] = nothing` |
| `delete!(s, x)` | `delete!(s.dict, x)` |
| `x in s` | `haskey(s.dict, x)` |
| `length(s)` | `length(s.dict)` |
| `for x in s` | iterates over the keys of `s.dict` |

Because `Set` is a `struct`, not a `mutable struct`, its one field can never be made to point to a different `Dict`. Using `push!` does not contradict the immutability of the struct. What it does is add a key to the `Dict` the field already points to.

```julia
d = s.dict
push!(s, 3)
s.dict === d           # true: the same Dict, now with the key 3
```

So `ismutable(s)` is `false`, because it asks only whether the `Set` object itself can be changed, that is, whether its own field can be reassigned. The contents of the set can still change. The docstring of `Set` in Base calls sets "mutable containers" in this second sense, the sense of the column "Can be changed" in the collections table above.

`s.dict` is an internal field, used here only to show how a set is stored. Code that works with sets should use `push!`, `delete!`, `in` and the other set functions, never the field.

All run on Julia 1.13.3.

#### `Pair`s

Below are some details about `Pair`s created by Claude and edited by me.

##### What a `Pair` is

A `Pair` is not a small `Dict`; it is what a `Dict` is made of. A dictionary is a collection of pairs together with fast lookup by key: its element type is `Pair`, and iterating over it hands back pairs. A single `Pair` has no lookup at all. It holds two values, `first` and `second`, which can be read by name or by position, and it cannot be changed once it is made.

So a `Pair` is closer to a tuple of two elements whose positions have names: `Tuple("a" => 1)` gives `("a", 1)`. Like a tuple, its two halves keep their own types (`Pair{String, Int64}`). The arrow `=>` suggests "key to value", and that is how `Dict` uses it, but a pair by itself means no more than "these two values, in this order".

The table was created assuming `p = "a" => 1` and `d = Dict("a" => 1)`:

| Call | Result | What it shows |
|---|---|---|
| `eltype(d)` | `Pair{String, Int64}` | the elements of a dictionary are pairs |
| `first(d) == p` | `true` | iterating over `d` gives back `"a" => 1` |
| `d["a"]` | `1` | a `Dict` looks a value up by its key |
| `p["a"]` | `MethodError` | a `Pair` has no keys, just two values |
| `p.first`, `p.second` | `"a"`, `1` | the two values, by name |
| `p[1]`, `p[2]` | `"a"`, `1` | or by position |
| `p.second = 2` | error: immutable struct of type `Pair` cannot be changed | a `Pair` is fixed once made |
| `Tuple(p)` | `("a", 1)` | the same two values as a tuple |
| `Dict(p)` | `Dict("a" => 1)` | a dictionary with one entry is built from a pair |

`=>` also appears outside dictionaries, wherever a function needs two values that belong together: `replace("abc", "a" => "x")` gives `"xbc"`, "replace this with that", with no dictionary involved.

All run on Julia 1.13.3.


##### When to use a `Pair` and when a `Tuple`

Use a `Pair` when two values stand in a *relation* with a direction: e.g. this key to that value, this index to that entry, replace this with that. The arrow indicates which side is which, and functions that expect such a relation need a `Pair` (e.g. `replace` in the table below).

Use a `Tuple` for a fixed group (of any length) of values, with no direction between these values: e.g. the several results of one function, coordinates, the arguments to pass to a function. Functions that return more than one value return a tuple, which is then taken apart by assignment.

The two overlap more than the arrow suggests: a pair can be taken apart like a tuple (`k, v = "a" => 1`), and `Dict` also accepts tuples of two values (`Dict([("a", 1), ("b", 2)])` gives `Dict("a" => 1, "b" => 2)`). Choosing `=>` is then a signal to the _reader_ that the two values form a relation.

When the positions deserve names, use a `NamedTuple` rather than a `Pair`: `first` and `second` only say which came first.

| Situation | Use | Example |
|---|---|---|
| key and value | `Pair` | `Dict("a" => 1)`, and `Dict(x => x^2 for x in 1:3)` |
| replace this with that | `Pair` | `replace([1, 2, 1], 1 => 0)` gives `[0, 2, 0]` |
| replace this with that, given as a tuple | `Pair` only | `replace("abc", ("a", "x"))` is a `MethodError` |
| index and entry | `Pair` | `collect(pairs([10, 20]))` gives `[1 => 10, 2 => 20]` |
| several results of one function | `Tuple` | `divrem(7, 2)` gives `(3, 1)`; `q, r = divrem(7, 2)` |
| largest value and where it is | `Tuple` | `findmax([3, 9, 4])` gives `(9, 2)` |
| more than two values | `Tuple` | `(1, "a", 2.0)` |
| arguments to pass on | `Tuple` | `t = (1, 2)`, then `f(t...)` calls `f(1, 2)` |
| values with meaningful names | `NamedTuple` | `(x = 1, y = 2)`, then `.y` gives `2` |

All run on Julia 1.13.3.


#### Static vectors: `SVector` and `MVector`

Below are some details about Static vectors created by Claude and edited by me.

##### What `SVector` and `MVector` are

`SVector` and `MVector` come from the `StaticArrays.jl` package. The *static* in its name does not mean "unchangeable". It means the **length is fixed and is part of the type**: `SVector(1, 2, 3)` is an `SVector{3, Int64}`, and `MVector(1, 2, 3)` an `MVector{3, Int64}`. The package README says so directly:

> "statically sized" means that the size can be determined from the *type*, and "static" does **not** necessarily imply `immutable`.

So both kinds are static. They differ only in whether their values can change: an `SVector` is an immutable `struct`, an `MVector` a `mutable struct`. An `MVector` is a static vector whose values can change, but whose length still cannot.

| Property | `SVector` | `MVector` |
|---|---|---|
| Length fixed, part of the type | yes | yes |
| Change a value: `v[2] = 10` | no: `setindex!` is not defined | yes: gives `[1, 10, 3]` |
| Grow: `push!(v, 4)` | no | no: `MethodError` (no `resize!`) |
| `ismutable(v)` | `false` | `true` |
| `isbits(v)` | `true` | `false` |

To get a changed `SVector`, build a new one: `Base.setindex(s, 10, 2)` (no `!`) returns a new `SVector` `[1, 10, 3]` and leaves `s` as it was. `SVector(m)` and `MVector(s)` convert between the two kinds.

All run on Julia 1.13.3 (StaticArrays 1.9.24). Quotes from the StaticArrays README and documentation ([index](https://juliaarrays.github.io/StaticArrays.jl/stable/), [API: mutable arrays](https://juliaarrays.github.io/StaticArrays.jl/stable/api/)).

##### Implementation details of `SVector` and `MVector` and when to use each

Why have a fixed length at all in these static arrays? Because the length is in the type, the package's functions know this length when the code is compiled. The StaticArrays documentation explains why that is useful for small arrays (up to about 100 elements):

> [Fixed length allows c]ompletely unrolling to loop-free code. e.g. `v3 = v1 + v2` is just implemented as 3 scalar additions, with no loop overhead at all, if these are all `SVector{3, Float64}`.

The `isbits` row in the table is the implementation detail behind the difference between the two types: an `SVector` of numbers is a plain value, like a number, so it needs no separately allocated memory of its own, "[a]voiding heap (or even stack) allocations of temporary arrays."

By contrast `MVector`s according to the documentation "live on the heap, not the stack." They can still be faster than a `Vector` for small sizes, because their fixed length "allows for additional loop unrolling and inlining".

To sum up: an `SVector` suits small fixed-size values that are computed rather than edited, such as points in 2D or 3D; an `MVector` suits a small fixed-size vector that is filled in or updated entry by entry.

Quotes from the StaticArrays documentation ([index](https://juliaarrays.github.io/StaticArrays.jl/stable/), [API: mutable arrays](https://juliaarrays.github.io/StaticArrays.jl/stable/api/)).

#### Destructuring in Julia

Below are some details about destructuring created by Claude and edited by me.

##### What destructuring is

Destructuring pulls several values out of one value and assigns them to several variables in a single statement. Julia has two kinds, and they work differently: one goes by position, the other by name.

| Aspect | By position | By name |
|---|---|---|
| Syntax | `a, b = v` | `(; x0, x1) = v` |
| How it works | iterates `v` | reads `v.x0` and `v.x1` |
| Works on | anything iterable | anything with fields of those names |
| Order of the variables | matters | does not matter |

A named tuple is one case where both work, since it is iterable and also has named fields.

Everything below was run on Julia 1.13.3. The manual covers this in the Functions chapter, under "Destructuring Assignment and Multiple Return Values".

##### Destructuring by position

Each variable receives the next value that iteration hands out:

```julia
a, b = [10, 20]                     # a = 10, b = 20
a, b = (10, "s")                    # a = 10, b = "s"
a, b = "xy"                         # a = 'x', b = 'y', two Chars
a, b = Dict(:p => 1, :q => 2)       # a and b are Pairs, not values
```

The number of variables need not match the number of values, as long as there are enough values:

```julia
a, b = 1:3                          # a = 1, b = 2, the 3 is dropped
a, b, c = [10, 20]                  # BoundsError: there is no third value
```

Some variations:

```julia
a, b... = [10, 20, 30]              # a = 10, b = [20, 30], the rest
a, _, c = (1, 2, 3)                 # a = 1, c = 3, _ skips a value
a, (b, c) = (1, (2, 3))             # a = 1, b = 2, c = 3, nested
a, b = b, a                         # swaps a and b
```

This is how you can extract the different parts of the value a function returns. `extrema([3, 1, 2])` returns one tuple, `(1, 3)`, and `lo, hi = extrema([3, 1, 2])` takes it apart (so `lo = 1` and `hi = 3`).


##### Destructuring by name

Each variable receives the field with the same name. The manual: "Instead of destructuring based on iteration, the right side of assignments can also be destructured using property names." A *property* is what `v.name` reads; for most values it is simply a field.

```julia
nt = (x0 = 1, x1 = 4, y0 = 2, y1 = 6)
(; x0, x1) = nt                     # x0 = 1, x1 = 4, the other fields are ignored
(; x1, x0) = nt                     # the same values: order does not matter
(; z) = nt                          # FieldError: no field z
```

The variable must have the same name as the field; `(; x0 => a) = nt` is a syntax error. This destructuring is useful because it allows you to use these individual fields succinctly in subsequent expressions. You can write `x0` instead of `nt.x0`, which is useful when `x0` is referenced multiple times.

It works on any value with named fields, not only on named tuples, and the value need not be iterable:

```julia
struct Point2; x; y; end
(; y) = Point2(7, 8)                # y = 8, and a Point2 is not iterable
(; second) = "a" => 1               # second = 1
z = 3 + 4im
(; re) = z                          # re = 3
```

It fails on a vector, a dictionary and a plain tuple, because none of them has a field called `x0`. A dictionary's key is not a field:

```julia
(; x0) = [1, 2]                     # FieldError
(; x0) = Dict(:x0 => 1)             # FieldError
(; x0) = (1, 2)                     # FieldError
```

Destructuring assigns ordinary variables, so a field's name can hide something you still need. `im` is Julia's imaginary unit, and it is also a field name of a complex number:

```julia
2im                     # 0 + 2im
(; re, im) = 1 + 2im    # re = 1, im = 2
im                      # 2, no longer the imaginary unit
2im                     # 4
im = Base.im            # restores it
2im                     # 0 + 2im
```

Inside a `let` or a function, the same line throws an error, because the `im` on the right is already the new local variable:

```julia
let
    (; re, im) = 1 + 2im    # UndefVarError: `im` not defined in local scope
end
```

To stay safe, destructure only the fields whose names are not already in use. Here that means leaving out `im`, which would replace the imaginary unit. Get that part of `z` with `imag(z)` and then explicitly multiply by `im`:

```julia
z = 1 + 2im
(; re) = z              # re = 1
imag(z)                 # 2
re + imag(z)            # 3, not z
re + imag(z)*im         # 1 + 2im, which is z
```


##### Warning: by position on a named tuple

Without the `;` the names are ignored and no error warns you:

```julia
nt = (x0 = 1, x1 = 4, y0 = 2, y1 = 6)
x1, x0 = nt                         # x1 = 1, x0 = 4: swapped
(; x1, x0) = nt                     # x1 = 4, x0 = 1: correct
```

So when a value has names, destructure by name.


##### The reverse: building a named tuple

On the right of an assignment, the same syntax builds a named tuple from variables, using their names as the field names:

```julia
x0 = 5
x1 = 6
p = (; x0, x1)                      # (x0 = 5, x1 = 6)
```


##### Destructuring in function arguments and loops

Both kinds of destructuring also work in a function's argument list. Sometimes you might get a result back as a tuple and you can then destructure by position to use it as is, without taking it apart first yourself. See the `add` example below. The extra parentheses matter: they make it one argument that is taken apart.

In general the one-argument form below is for values that already come packaged together, such as the entries of a collection of tuples, or a tuple a function returns:

```julia
add((x, y)) = x + y
add2(x, y) = x + y

points = [(1, 2), (3, 4)]
map(add, points)                    # [3, 7]
map(add2, points)                   # MethodError: map hands add2 one tuple, not two values

add(extrema([3, 1, 2]))             # 4
add2(extrema([3, 1, 2]))            # MethodError, for the same reason
add2(extrema([3, 1, 2])...)         # 4, the ... splits the tuple into two arguments
```

Destructuring by name works the same way:

```julia
slope((; x0, x1, y0, y1)) = (y1 - y0) / (x1 - x0)
slope(nt)                           # 1.3333333333333333
```

This form suits the `f(x, p)` style, where a named tuple carries a function's parameters.

Both types of destructuring also work in a `for` loop:

```julia
for (k, v) in Dict("a" => 1)        # by position, from each Pair
    println(k, " ", v)              # a 1
end

for (i, x) in enumerate(["p", "q"]) # by position, from each (index, value) tuple
    println(i, " ", x)              # 1 p, then 2 q
end

for (; x0, x1) in [nt]              # by name
    println(x0, " ", x1)            # 1 4
end
```